# Activation Functions & Initialization — AI Lab Instructor Notebook

*Deep Learning · Hard*



5 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import math
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Activation Functions & Initialization — Student Lab

Focus: implement activations + init schemes and empirically verify signal/gradient propagation across depth.

## Section 0 — Setup
We’ll work with synthetic Gaussian inputs so we can isolate activation/init effects.

In [ ]:
import math
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

n = 4096
d = 256
X0 = rng.normal(0.0, 1.0, size=(n, d)).astype(float)
check('X0_shape', X0.shape == (n, d))
print('X0 mean', float(X0.mean()), 'std', float(X0.std()))

**Why this works.** # Activation Functions & Initialization — Instructor Lab

Solutions + short rationales. Focus: variance preservation (Xavier/He) and how activations impact forward/gradient propagation.

## Task 2: Implement ReLU, tanh, and GELU (approx).
*Section: Activations*

## Section 1 — Activations

### Task 1.1
Implement ReLU, tanh, and GELU (approx).

### Task 1.2
Compare output mean/std on standard normal input.

### Optional concept — Softplus

`softplus(v) = log(1 + e^v)`

Why it is used:
- smooth approximation of ReLU
- avoids numerical instability with a stable implementation
- useful in logistic-loss derivations

Behavior:
- if `v` is very large positive, `log(1 + e^v) ≈ v`
- if `v` is very large negative, `log(1 + e^v) ≈ 0`

So Softplus behaves like `max(0, v)` but smoothly.

In [ ]:
def relu(x: np.ndarray) -> np.ndarray:
    return np.maximum(0.0, x)

def tanh(x: np.ndarray) -> np.ndarray:
    return np.tanh(x)

def gelu(x: np.ndarray) -> np.ndarray:
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * (x**3))))

for name, f in [('relu', relu), ('tanh', tanh), ('gelu', gelu)]:
    Y = f(X0)
    print(name, 'mean', float(Y.mean()), 'std', float(Y.std()), 'min', float(Y.min()), 'max', float(Y.max()))

check('relu_nonneg', (relu(X0) >= 0).all())

In [ ]:
# Checks
check('relu_nonneg', (relu(X0) >= 0).all())

**Why this works.** ## Section 1 — Activations

Rationale: activations change the distribution (mean/variance) and the local derivative distribution, both of which affect training stability.

Softplus reminder:
- `softplus(v) = log(1 + e^v)` is a smooth ReLU surrogate
- in practice use stable form: `max(v, 0) + log1p(exp(-abs(v)))`

## Task 3: Implement:
*Section: Initialization schemes*

## Section 2 — Initialization schemes

We’ll initialize weight matrices for a linear layer: `Y = X W` (no bias).

### Task 2.1
Implement:
- naive normal init with std=1
- Xavier normal init
- He normal init

Return a weight matrix of shape (fan_in, fan_out).

In [ ]:
def init_naive(fan_in: int, fan_out: int, seed=0) -> np.ndarray:
    r = np.random.default_rng(seed)
    return r.normal(0.0, 1.0, size=(fan_in, fan_out))

def init_xavier_normal(fan_in: int, fan_out: int, seed=0) -> np.ndarray:
    r = np.random.default_rng(seed)
    std = math.sqrt(2.0 / (fan_in + fan_out))
    return r.normal(0.0, std, size=(fan_in, fan_out))

def init_he_normal(fan_in: int, fan_out: int, seed=0) -> np.ndarray:
    r = np.random.default_rng(seed)
    std = math.sqrt(2.0 / fan_in)
    return r.normal(0.0, std, size=(fan_in, fan_out))

Wn = init_naive(d, d, seed=0)
Wx = init_xavier_normal(d, d, seed=0)
Wh = init_he_normal(d, d, seed=0)

print('naive W std', float(Wn.std()))
print('xavier W std', float(Wx.std()))
print('he    W std', float(Wh.std()))
check('W_shapes', Wn.shape == (d,d) and Wx.shape == (d,d) and Wh.shape == (d,d))

In [ ]:
# Checks
check('W_shapes', Wn.shape == (d,d) and Wx.shape == (d,d) and Wh.shape == (d,d))

**Why this works.** ## Section 2 — Initialization schemes

Rationale: for `Y = XW` with iid assumptions, you can choose Var(W) so that Var(Y) ≈ Var(X).
- Xavier targets tanh-like symmetric activations (preserve variance both directions)
- He targets ReLU-like activations (half the mass is zero → compensate with 2/fan_in)

## Task 4: Write `simulate_forward(X0, L, init_fn, act_fn)` returning stats per layer.
*Section: Forward signal propagation across depth*

## Section 3 — Forward signal propagation across depth

We simulate an L-layer network: `X_{l+1} = act(X_l W_l)`

### Task 3.1
Write `simulate_forward(X0, L, init_fn, act_fn)` returning stats per layer.

We care about:
- mean/std of activations
- for ReLU: fraction of zeros
- for tanh: saturation (`|a| > 0.95`) and average local derivative

### Task 3.2
Compare naive vs Xavier/He for depth L=50 using both ReLU and tanh.

In [ ]:
$10

**Why this works.** ## Section 3 — Forward signal propagation

Rationale: deep networks fail when activations explode (std grows) or vanish (std → 0). Proper init keeps the signal roughly stable across depth.

Important: for tanh, std alone is not enough. Also inspect saturation (`|a|>0.95`) and average derivative; a high std can still mean heavy saturation and poor gradient flow.

## Task 5: Implement activation derivatives for ReLU and tanh.
*Section: Backward gradient propagation (toy)*

## Section 4 — Backward gradient propagation (toy)

We estimate gradient flow using a simple scalar loss:
- Forward: X_{l+1} = act(X_l W_l)
- Loss: mean(X_L)
- Backward (approx): propagate gradients using local Jacobians

This is not a full autodiff engine; it’s a controlled experiment to see gradient norms explode/vanish.

### Task 4.1
Implement activation derivatives for ReLU and tanh.

### Task 4.2
Simulate gradient norms across depth for different init schemes.

In [ ]:
$11

In [ ]:
# Checks
check('drelu_binary', set(np.unique(drelu(np.array([-1.0, 0.0, 2.0])))).issubset({0.0, 1.0}))

**Why this works.** ## Section 4 — Backward gradient propagation (toy)

Rationale: even if forward activations look reasonable, gradients can still vanish/explode. This toy Jacobian experiment gives a quick proxy.

Interpretation:
- Vanishing gradients: repeated Jacobian factors with norm below 1 drive early-layer gradients toward 0, so those layers learn very slowly.
- Exploding gradients: repeated Jacobian factors with norm above 1 blow gradient norms up, which destabilizes optimization and can produce divergence or NaNs.